[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C34_Agent_Orchestration_Course/04_observability/04_observability.ipynb)

# 04 · 可观测与评测（Observability）

目标：用**纯标准库**从零搭一套 agent 可观测——**span/trace 树 → 结构化日志 → 成本/延迟/token 聚合 → 评测**，全程 `assert` 验证，**无需 API key**。

路线：span 数据模型 → with-span 上下文管理器(自动父子+起止) → 重建 span 树 → 结构化日志(过滤/聚合) → 在叶 span 挂成本、沿树聚合 → 看客观结果的评测 + pass^k → 给编排织 trace → ✏️ 练习 → 📖 答案 → 🧪 真实 trace 形状胶囊。

> 心智模型：**可观测 = 给一次跨多 agent 的运行拍一张可回看的 X 光片(span 树)**。读懂这棵树就能定位错误、算清成本、找出瓶颈、断言对错。

## 1 · span 数据模型 + with-span 上下文管理器

span = 一个工作单元，带 `{trace_id, span_id, parent_id, name, start, end, attributes}`。
用上下文管理器 `with tracer.span(name):` 自动记父子关系与起止时间——无侵入、且**异常也保证闭合**。

In [ ]:
import time, json, contextlib
from collections import defaultdict

class Tracer:
    '''最小 tracer：用调用栈追踪『当前在哪个 span 里』，自动建父子关系。'''
    def __init__(self, trace_id='T1', clock=None):
        self.trace_id = trace_id
        self.spans = []                       # 所有 span(扁平)
        self._stack = []                      # 当前 span_id 栈
        self._next_id = 0
        self._clock = clock or time.perf_counter   # 可注入时钟(测试用确定性)
    @contextlib.contextmanager
    def span(self, name, **attrs):
        self._next_id += 1
        sid = self._next_id
        parent = self._stack[-1] if self._stack else None
        rec = {'trace_id': self.trace_id, 'span_id': sid, 'parent_id': parent,
               'name': name, 'start': self._clock(), 'end': None,
               'attributes': dict(attrs)}
        self.spans.append(rec)
        self._stack.append(sid)
        try:
            yield rec                         # 业务代码在这里跑；可往 rec['attributes'] 挂数据
        finally:
            rec['end'] = self._clock()        # 异常也保证记 end、出栈(span 必闭合)
            self._stack.pop()

# 用一个假时钟(每次 +1)让起止时间确定、可断言
_t = [0]
def fake_clock():
    _t[0] += 1; return _t[0]

tr = Tracer(clock=fake_clock)
with tr.span('root') as root:
    with tr.span('child-A'):
        pass
    with tr.span('child-B'):
        with tr.span('grandchild'):
            pass
for s in tr.spans:
    print(f"id={s['span_id']} parent={s['parent_id']} name={s['name']:11s} [{s['start']}..{s['end']}]")
# root 无父；child 的父是 root；grandchild 的父是 child-B
byname = {s['name']: s for s in tr.spans}
assert byname['root']['parent_id'] is None
assert byname['child-A']['parent_id'] == byname['root']['span_id']
assert byname['grandchild']['parent_id'] == byname['child-B']['span_id']
assert all(s['end'] is not None for s in tr.spans)   # 全部闭合
print('✅ tracer：with-span 自动记父子+起止；4 个 span 全部正确闭合')

## 2 · span 必闭合：异常也不漏 end

一个经典陷阱是手动 start/end 时异常导致 end 没记上、span 永远『开着』。
上下文管理器的 `finally` 保证**无论是否异常**，end 一定记上、栈一定弹出。

In [ ]:
_t[0] = 0
tr2 = Tracer(clock=fake_clock)
raised = False
try:
    with tr2.span('outer'):
        with tr2.span('boom'):
            raise ValueError('span 内部炸了')
except ValueError:
    raised = True
assert raised
# 即使 boom 内部抛异常: boom 与 outer 都被正确闭合, 栈被清空
assert all(s['end'] is not None for s in tr2.spans), '异常也必须记 end'
assert tr2._stack == [], '异常后调用栈必须清空(否则后续 span 父子全错)'
print('span 数:', len(tr2.spans), '| 全部闭合:', all(s['end'] for s in tr2.spans), '| 栈空:', tr2._stack == [])
print('✅ 异常安全：finally 保证 span 必闭合、栈必清空 —— 树不会建坏')

## 3 · 重建 span 树 + 遍历

扁平的 span 列表靠 `parent_id` 拼成树（真实里 span 来自不同进程、乱序到达，靠三个 id 拼回）。
实现建树 + 前序打印（缩进体现层级）。

In [ ]:
def build_tree(spans):
    '''按 parent_id 建 children 索引；返回 (根 span_id 列表, children dict)。'''
    children = defaultdict(list)
    roots = []
    for s in spans:
        if s['parent_id'] is None:
            roots.append(s['span_id'])
        else:
            children[s['parent_id']].append(s['span_id'])
    return roots, children

def print_tree(spans):
    byid = {s['span_id']: s for s in spans}
    roots, children = build_tree(spans)
    lines = []
    def walk(sid, depth):
        s = byid[sid]
        lines.append('  ' * depth + f"[{s['name']}]")
        for ch in children[sid]:
            walk(ch, depth + 1)
    for r in roots:
        walk(r, 0)
    return lines

for line in print_tree(tr.spans):
    print(line)
roots, children = build_tree(tr.spans)
assert len(roots) == 1                                  # 一个根
assert len(children[byname['root']['span_id']]) == 2    # root 有 2 个孩子(A、B)
assert len(children[byname['child-B']['span_id']]) == 1 # child-B 有 1 个孙(grandchild)
print('✅ span 树重建：扁平 span 列表 -> 树；结构正确可遍历')

## 4 · 结构化日志：过滤 + 聚合 + 关联 span

日志要**结构化**(机器可解析字段)而非自由文本：带 `level / event / span_id / payload`。
好处：按 level 过滤、按 event 聚合、靠 span_id 挂回 span 树。

In [ ]:
class StructLogger:
    LEVELS = {'debug': 10, 'info': 20, 'warn': 30, 'error': 40}
    def __init__(self):
        self.records = []
    def log(self, level, event, span_id=None, **payload):
        self.records.append({'level': level, 'event': event,
                             'span_id': span_id, **payload})
    def filter(self, min_level):
        '''只保留 >= min_level 的(生产常只看 warn 以上)。'''
        thr = self.LEVELS[min_level]
        return [r for r in self.records if self.LEVELS[r['level']] >= thr]
    def count_by_event(self):
        '''按 event 聚合计数(如统计重试/越权各几次)。'''
        agg = defaultdict(int)
        for r in self.records:
            agg[r['event']] += 1
        return dict(agg)

log = StructLogger()
log.log('info', 'subagent_start', span_id=3, agent='A')
log.log('debug', 'cache_hit', span_id=3)
log.log('warn', 'retry', span_id=3, attempt=1)
log.log('error', 'subagent_failed', span_id=4, agent='B')
log.log('warn', 'retry', span_id=4, attempt=1)

warns = log.filter('warn')                 # 只看 warn 以上
print('warn+ 条数:', len(warns))
print('按 event 聚合:', log.count_by_event())
assert len(warns) == 3                      # 2 retry + 1 error
assert log.count_by_event()['retry'] == 2   # retry 出现 2 次
# 关联 span: span 3 上发生的所有事件
span3_events = [r['event'] for r in log.records if r['span_id'] == 3]
assert span3_events == ['subagent_start', 'cache_hit', 'retry']
print('span 3 上的事件:', span3_events)
print('✅ 结构化日志：按 level 过滤、按 event 聚合、靠 span_id 挂回 span 树')

## 5 · 成本/token 挂在叶 span、沿树聚合

在叶 span 的 attributes 挂 `tokens / cost`，沿树**后序遍历**聚合：
`cost(span) = self_cost + Σ cost(children)`。根聚合出总账；可下钻看哪个最贵。

In [ ]:
# 单价(每 1000 token 的钱)，不同模型不同
PRICE = {'opus': 0.015, 'haiku': 0.001}

def leaf_cost(tokens, model):
    return tokens / 1000 * PRICE[model]

# 造一棵带成本的 trace: root -> 两个 subagent，各含一次 llm_call(挂 tokens/cost)
_t[0] = 0
tc = Tracer(clock=fake_clock)
with tc.span('root'):
    with tc.span('subagent-A'):
        with tc.span('llm_call', model='opus') as s:
            s['attributes']['tokens'] = 8000
            s['attributes']['cost'] = leaf_cost(8000, 'opus')
    with tc.span('subagent-B'):
        with tc.span('llm_call', model='haiku') as s:
            s['attributes']['tokens'] = 2000
            s['attributes']['cost'] = leaf_cost(2000, 'haiku')

def aggregate(spans, key):
    '''沿树后序聚合某个数值属性(自身 + 子树之和)。返回 {span_id: 子树总和}。'''
    byid = {s['span_id']: s for s in spans}
    _, children = build_tree(spans)
    total = {}
    def post(sid):
        s = byid[sid]
        own = s['attributes'].get(key, 0)
        total[sid] = own + sum(post(ch) for ch in children[sid])
        return total[sid]
    roots, _ = build_tree(spans)
    for r in roots:
        post(r)
    return total

byname2 = {s['name']: s for s in tc.spans}
cost_total = aggregate(tc.spans, 'cost')
tok_total = aggregate(tc.spans, 'tokens')
root_id = byname2['root']['span_id']
print(f"总 token={tok_total[root_id]} | 总成本=${cost_total[root_id]:.4f}")
# 下钻: 各 subagent 的成本
a_id, b_id = byname2['subagent-A']['span_id'], byname2['subagent-B']['span_id']
print(f"  subagent-A=${cost_total[a_id]:.4f} | subagent-B=${cost_total[b_id]:.4f}")
assert tok_total[root_id] == 10000                       # 8000+2000
assert abs(cost_total[root_id] - (0.12 + 0.002)) < 1e-9  # opus 贵、haiku 便宜
assert cost_total[a_id] > cost_total[b_id]               # A(opus) 比 B(haiku) 贵
print('✅ 成本聚合：叶上挂数、沿树累加；总账正确、可下钻找最贵的枝')

## 6 · 评测：看客观结果 + trace 断言 + pass^k

**别信 agent 自述，看客观判据**：结果断言(含所有要求部分)、过程断言(在 trace 上，如派了几个 subagent、无越权)、可靠性(pass^k)。

In [ ]:
def eval_result(report, required_parts):
    '''结果断言: 最终报告是否客观包含所有要求的部分(而非 agent 说做完了).'''
    missing = [p for p in required_parts if p not in report]
    return {'passed': len(missing) == 0, 'missing': missing}

def eval_trace(spans, expected_subagents):
    '''过程断言(在 trace 上): 是否派了正确数量的 subagent.'''
    n_sub = sum(1 for s in spans if s['name'].startswith('subagent'))
    return {'passed': n_sub == expected_subagents, 'n_subagents': n_sub}

def pass_pow_k(success_rate, k):
    '''pass^k: 连续 k 次全部成功的概率 ≈ p^k(衡量可靠性).'''
    return success_rate ** k

# 评一份『三公司对比报告』
good_report = '对比：A公司 +10%；B公司 +12%；C公司 -3%'
bad_report = '对比：A公司 +10%；B公司 +12%'              # 漏了 C公司!
assert eval_result(good_report, ['A公司', 'B公司', 'C公司'])['passed'] is True
r_bad = eval_result(bad_report, ['A公司', 'B公司', 'C公司'])
assert r_bad['passed'] is False and r_bad['missing'] == ['C公司']  # 客观查出漏项
# 过程: 我们前面 tc 这棵树派了 2 个 subagent
assert eval_trace(tc.spans, expected_subagents=2)['passed'] is True
# 可靠性: 单步 0.9 的多 agent，5 步连乘只剩 ~0.59
assert abs(pass_pow_k(0.9, 5) - 0.59049) < 1e-5
print('结果评测(漏C):', r_bad)
print('pass^5 @ p=0.9 =', round(pass_pow_k(0.9, 5), 3), '(连乘让可靠性衰减)')
print('✅ 评测：看客观结果(查出漏项) + trace 过程断言 + pass^k 可靠性')

---
## ✏️ 练习 1：找出最慢的**叶** span（瓶颈定位）

可观测的常见用途：找瓶颈。注意——根 / 中间 span 的 `end-start` 含了子节点的时间，跨层比较会误导；
真正的瓶颈在**叶 span**（真实开销发生处）。

实现 `slowest_leaf(spans)`：只在**叶 span**（没有任何 span 以它为 parent）里，按 `end-start` 返回耗时最长的 span 的 `name`。

In [ ]:
def slowest_leaf(spans):
    # TODO: 1) 找出所有叶 span(其 span_id 不是任何 span 的 parent_id)
    #       2) 在叶里按 end-start 返回耗时最长者的 name
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
spans = [
    {'span_id': 1, 'parent_id': None, 'name': 'root', 'start': 0, 'end': 100},   # 非叶(下面有子)
    {'span_id': 2, 'parent_id': 1, 'name': 'fast', 'start': 10, 'end': 15},   # 叶, 5
    {'span_id': 3, 'parent_id': 1, 'name': 'slow', 'start': 20, 'end': 90},   # 叶, 70 最慢
    {'span_id': 4, 'parent_id': 1, 'name': 'mid', 'start': 90, 'end': 100},   # 叶, 10
]
assert slowest_leaf(spans) == 'slow'      # root 虽 end-start=100 但非叶, 不参与
print('最慢的叶 span:', slowest_leaf(spans))
print('✅ 练习 1 通过：在叶 span 里定位真实瓶颈(跨层比 end-start 会误导)')

## ✏️ 练习 2：并行子 span 的延迟聚合（max 而非 sum）

**串行**子 span 延迟相加；**并行**子 span(fan-out 同时跑)延迟取 **max**。

实现 `parallel_latency(child_durations)`：给定一组并行子 span 的耗时列表，返回它们的**总延迟**(=最大值，因为同时跑)；空列表返回 0。

In [ ]:
def parallel_latency(child_durations):
    # TODO: 并行子 span 同时跑 -> 总延迟 = max(各耗时)；空 -> 0
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert parallel_latency([30, 50, 20]) == 50    # 三个并行 subagent, 等最慢的 50
assert parallel_latency([]) == 0
assert parallel_latency([7]) == 7
# 对比: 若串行则是 sum=100；并行只要 50 —— 这就是 fan-out 提速的来源
print('3 个并行子 span 总延迟:', parallel_latency([30, 50, 20]), '(串行会是 100)')
print('✅ 练习 2 通过：并行子 span 延迟取 max，体现 fan-out 的提速')

## ✏️ 练习 3：在 trace 上做安全过程断言（无越权）

把模块 03 的安全接进评测：检查一次运行的 trace 里**没有越权事件**。

实现 `eval_no_escalation(log_records)`：给定结构化日志，若**没有**任何 `event=='privilege_escalation'` 的记录返回 `{'passed':True,'violations':[]}`；否则 `passed=False`、`violations` 为这些记录里的 `tool` 字段列表。

In [ ]:
def eval_no_escalation(log_records):
    # TODO: esc = [r for r in log_records if r['event']=='privilege_escalation']
    #   返回 {'passed': len(esc)==0, 'violations': [r['tool'] for r in esc]}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
clean_log = [{'event': 'subagent_start', 'tool': None},
             {'event': 'tool_call', 'tool': 'web_search'}]
dirty_log = clean_log + [{'event': 'privilege_escalation', 'tool': 'send_email'}]
assert eval_no_escalation(clean_log)['passed'] is True
out = eval_no_escalation(dirty_log)
assert out['passed'] is False and out['violations'] == ['send_email']
print('含越权的运行评测:', out)
print('✅ 练习 3 通过：在 trace/日志上断言『本次运行无越权』(安全∩可观测)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def slowest_leaf(spans):
    parents = {s.get('parent_id') for s in spans}        # 所有出现过的 parent_id
    leaves = [s for s in spans if s['span_id'] not in parents]   # 没人以它为父 = 叶
    return max(leaves, key=lambda s: s['end'] - s['start'])['name']

In [ ]:
# 练习 2 参考答案
def parallel_latency(child_durations):
    return max(child_durations) if child_durations else 0

In [ ]:
# 练习 3 参考答案
def eval_no_escalation(log_records):
    esc = [r for r in log_records if r['event'] == 'privilege_escalation']
    return {'passed': len(esc) == 0, 'violations': [r['tool'] for r in esc]}

---
## 🧪 真实数据胶囊：OpenTelemetry / GenAI 风格的 trace 形状

真实里一次 LLM/agent 调用按 **OpenTelemetry GenAI 语义约定**记成 span，属性带模型名、输入/输出 token、成本等。

下面用**贴近真实**的 GenAI span 形状，端到端给一次「分解→fan-out→合成」编排织 trace，并产出账单 + 评测。

> 形状对照：真实里属性键如 `gen_ai.request.model`、`gen_ai.usage.input_tokens`；本课用简化键，结构一致。

In [ ]:
# 贴近真实 GenAI 约定的 span 属性
def genai_span(tr, name, model, in_tok, out_tok):
    with tr.span(name, **{'gen_ai.request.model': model,
                          'gen_ai.usage.input_tokens': in_tok,
                          'gen_ai.usage.output_tokens': out_tok}) as s:
        s['attributes']['tokens'] = in_tok + out_tok
        s['attributes']['cost'] = leaf_cost(in_tok + out_tok, model)
        return s

# 端到端: 给三公司研究编排织 trace
_t[0] = 0
rt = Tracer(trace_id='research-001', clock=fake_clock)
with rt.span('research: 三公司对比'):
    with rt.span('orchestrator: decompose'):
        genai_span(rt, 'llm_call', 'haiku', 200, 50)        # 分解用便宜模型
    for co, tok in [('A', 3000), ('B', 3500), ('C', 2800)]:
        with rt.span(f'subagent: 查{co}公司'):
            genai_span(rt, 'llm_call', 'opus', tok, 300)    # 检索用强模型
    with rt.span('orchestrator: synthesize'):
        genai_span(rt, 'llm_call', 'opus', 900, 200)        # 合成

# 账单
cost = aggregate(rt.spans, 'cost')
tok = aggregate(rt.spans, 'tokens')
rid = [s for s in rt.spans if s['parent_id'] is None][0]['span_id']
print(f"这次研究: {tok[rid]} token, ${cost[rid]:.4f}")
# 评测: 派了 3 个 subagent、span 全闭合
assert eval_trace(rt.spans, 3)['passed'] is True
assert all(s['end'] is not None for s in rt.spans)
assert tok[rid] == (250) + (3300 + 3800 + 3100) + (1100)   # 分解+3检索+合成
print('subagent 数:', eval_trace(rt.spans, 3)['n_subagents'])
print('✅ 复现 GenAI 风格 trace：编排织 span 树 -> 算账 -> 过程评测')

**🧪 胶囊练习**：实现 `model_cost_breakdown(spans)`：给定一棵 trace 的 span 列表，按 `gen_ai.request.model` 把各 LLM 调用的 `cost` 汇总成 `{模型名: 该模型总成本}`。（FinOps 里就这样看『钱花在哪个模型上』来决定能不能降级。）

In [ ]:
def model_cost_breakdown(spans):
    # TODO: 遍历带 'gen_ai.request.model' 属性的 span，按模型名累加它们的 'cost'
    #   返回 {model: 总 cost}
    raise NotImplementedError

In [ ]:
# 自测
bd = model_cost_breakdown(rt.spans)
print('按模型的成本:', {k: round(v, 4) for k, v in bd.items()})
assert set(bd) == {'haiku', 'opus'}
assert bd['opus'] > bd['haiku']      # opus 用得多又贵
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def model_cost_breakdown(spans):
    from collections import defaultdict
    agg = defaultdict(float)
    for s in spans:
        m = s['attributes'].get('gen_ai.request.model')
        if m is not None:
            agg[m] += s['attributes'].get('cost', 0)
    return dict(agg)

---
## 🔧 旁注：给真实 Claude 调用织 trace

本课用 `with tracer.span(...)` 织的 trace，换成真实 Claude 只是在 LLM 调用外包一层 span、把 `resp.usage` 挂上去（伪代码，**本环境不跑、需 API key；无 key 自动回退 MockLLM**）：

```python
import anthropic
client = anthropic.Anthropic()

def traced_llm(tracer, prompt, model='claude-opus-4-8'):
    with tracer.span('llm_call', **{'gen_ai.request.model': model}) as s:
        resp = client.messages.create(model=model, max_tokens=1024,
                                      messages=[{'role':'user','content':prompt}])
        s['attributes']['gen_ai.usage.input_tokens'] = resp.usage.input_tokens   # 真实 token!
        s['attributes']['gen_ai.usage.output_tokens'] = resp.usage.output_tokens
        s['attributes']['tokens'] = resp.usage.input_tokens + resp.usage.output_tokens
        s['attributes']['cost'] = leaf_cost(s['attributes']['tokens'], 'opus')
        return ''.join(b.text for b in resp.content if b.type=='text')
# 真实里直接用 OpenTelemetry SDK + GenAI instrumentation 自动织, 或发到 LangSmith/Langfuse
```

对应关系：`with tracer.span` ↔ OTel span、`resp.usage` ↔ 真实 token、`aggregate` 沿树聚合 ↔ 平台的成本下钻。你练的 span 树 / 聚合 / 评测逻辑**原样适用**，换成 OTel SDK 或 LangSmith 只是把『记录后端』换掉。这就是「scaffold 可迁移」。

### 小结
- 可观测 = 给一次跨多 agent 的运行拍一张**可回看的 X 光片(span 树)**。
- **span/trace/树**：用 trace_id/span_id/parent_id 拼树；`with span` 自动记父子+起止、**异常也闭合**。
- **结构化日志**(非自由文本)：按 level 过滤、按 event 聚合、靠 span_id 挂回树。
- **成本**：叶上挂 token/cost，沿树后序聚合；总账 + 可下钻找最贵的枝；并行延迟取 max。
- **评测**：别信自述、看**客观结果**(查漏项) + trace 过程断言(派几个/无越权) + **pass^k** 可靠性。
- 织进编排是**无侵入横切**：织一次，调试/算账/找瓶颈/评测/安全审计处处受益。

下一站：**模块 05 · 部署 Agent** —— 把这套受控、可观测的编排搬上生产：队列 + worker + 状态机 + 重试 + 持久 + 健康检查。